# Olist E-Commerce Business Intelligence

This notebook explores the cleaned star-schema tables produced by `python/data_cleaning.py`. The SQLite tables keep orders, item lines, payments, and reviews at separate grains to prevent fan-out double counting.

Run the ETL from the repository root before running this notebook.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DB = ROOT / 'data' / 'processed' / 'olist_analytics.db'
with sqlite3.connect(DB) as con:
    audit = pd.read_sql_query('SELECT * FROM data_quality_audit', con)
audit

## Monthly delivered sales
Product revenue below is the sum of item prices; freight and recorded payments are not included.

In [ ]:
with sqlite3.connect(DB) as con:
    monthly = pd.read_sql_query("""SELECT strftime('%Y-%m', o.order_purchase_timestamp) month,
      SUM(i.price) product_revenue, COUNT(DISTINCT o.order_id) orders
      FROM fact_orders o JOIN fact_order_items i USING(order_id)
      WHERE o.order_status='delivered' GROUP BY month ORDER BY month""", con)
monthly.plot(x='month', y='product_revenue', kind='line', marker='o', title='Monthly delivered product revenue', figsize=(12,4))
plt.xticks(rotation=45); plt.tight_layout()

## Category performance
Each item line contributes once to product revenue. Order counts are distinct orders per category.

In [ ]:
with sqlite3.connect(DB) as con:
    category = pd.read_sql_query("""SELECT category_english category, SUM(price) product_revenue,
      COUNT(DISTINCT order_id) orders FROM fact_order_items i
      JOIN fact_orders o USING(order_id) WHERE o.order_status='delivered'
      GROUP BY category ORDER BY product_revenue DESC LIMIT 15""", con)
category

## Delivery and customer experience
Delivery duration is calculated only where purchase and delivered timestamps exist. `delivery_late` compares actual delivery with the estimated delivery date and is defined only for delivered orders.

In [ ]:
with sqlite3.connect(DB) as con:
    delivery = pd.read_sql_query("""SELECT delivery_late, AVG(delivery_days) avg_delivery_days,
      COUNT(*) orders FROM fact_orders WHERE order_status='delivered'
      AND delivery_days IS NOT NULL GROUP BY delivery_late""", con)
delivery